# 🏷️ Meme Annotator — сырые картинки → схема проекта (устойчив к падениям)

Берёт папку с мемами и раскладывает в схему проекта
`{id, source, title, image_desc, meaning, caption_text}`.
Пайплайн на картинку: **OCR (easyocr)** → **VLM (Qwen2-VL)** → поля.

**Устойчивость к обрывам Colab:** результаты пишутся на Drive **чанками**
(`chunk_00001.zip`, ...). При повторном запуске уже обработанные картинки
пропускаются (готовые id читаются из записанных чанков). Отключился —
просто `Run all` заново, продолжит с места обрыва.

**Запуск:** `Runtime → Change runtime type → T4 GPU`, потом `Runtime → Run all`.


In [ ]:
# 1) Зависимости
!pip -q install "transformers>=4.45" accelerate bitsandbytes qwen-vl-utils easyocr Pillow tqdm

In [ ]:
# 2) Настройки
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # против фрагментации VRAM

INPUT_DIR   = "/content/drive/MyDrive/Memes"            # папка с сырыми картинками
OUT_DIR     = "/content/drive/MyDrive/Memes_annotated"  # куда писать чанки (переживает обрыв)
SOURCE      = "phone"                                    # тег источника: phone / imgflip / reddit / manual
MODEL_ID    = "Qwen/Qwen2-VL-7B-Instruct"
OCR_LANGS   = ["en"]                                    # языки easyocr
CHUNK_SIZE  = 100                                       # картинок на один чанк-zip (меньше = меньше потерь при обрыве)
MAX_IMAGES  = 100000                                    # ограничитель (для теста ставь 20)
DROP_UNSAFE = True                                      # выкидывать помеченные VLM как NSFW

# Ограничение числа vision-токенов Qwen2-VL — ГЛАВНЫЙ фикс OOM на больших картинках.
# Крупная картинка иначе даёт тысячи токенов и рвёт 16 ГБ T4. Понизь MAX_PIXELS, если всё ещё OOM.
MIN_PIXELS  = 256 * 28 * 28
MAX_PIXELS  = 768 * 28 * 28

In [ ]:
# 3) Смонтировать Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 4) Собрать список картинок
import os, glob

EXTS = ("*.jpg", "*.jpeg", "*.png", "*.webp", "*.gif", "*.JPG", "*.PNG", "*.JPEG")
paths = []
for e in EXTS:
    paths += glob.glob(os.path.join(INPUT_DIR, "**", e), recursive=True)
paths = sorted(set(paths))[:MAX_IMAGES]
assert paths, "В INPUT_DIR не нашлось картинок — проверь путь (папка Memes)."
print("картинок найдено:", len(paths))

In [ ]:
# 5) OCR-ридер (easyocr) на CPU — освобождает VRAM под VLM (важно на T4).
#    Медленнее на пару сек/картинку, но памяти под Qwen2-VL остаётся больше.
import easyocr
reader = easyocr.Reader(OCR_LANGS, gpu=False)
print("easyocr готов (CPU)")

In [ ]:
# 6) Загрузка VLM Qwen2-VL (4-bit). Самая долгая ячейка — один раз.
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type="nf4")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, torch_dtype=torch.float16, device_map="auto")
# min/max_pixels ограничивают разрешение картинки -> число vision-токенов -> память
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
print("VLM loaded on", model.device)

In [ ]:
# 7) Промпт (= протокол MemeCap) + разбор ответа + аннотация одной картинки
import json, re

def build_prompt(ocr):
    return (
        "You are annotating an internet meme for a search dataset. "
        "Reply with ONLY one JSON object, no extra text.\n"
        "Text detected on the image (OCR): \"" + (ocr or "") + "\"\n"
        "Produce these fields:\n"
        "- image_desc: literally what is shown in the image (characters, objects, scene). Do NOT explain the joke.\n"
        "- meaning: what the meme poster is trying to convey (the joke / message / context).\n"
        "- title: a very short title, 3-5 words.\n"
        "- safe: false if the image is sexual, gory, or hateful; otherwise true.\n"
        'Reply exactly like: {"title": "...", "image_desc": "...", "meaning": "...", "safe": true}'
    )

def _json(s):
    m = re.search(r"\{.*\}", s, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def annotate(path, ocr_text):
    # Одна тяжёлая картинка не должна ронять весь прогон: ловим OOM и чистим кэш.
    inputs = None
    try:
        messages = [{"role": "user", "content": [
            {"type": "image", "image": path},
            {"type": "text", "text": build_prompt(ocr_text)},
        ]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        imgs, vids = process_vision_info(messages)
        inputs = processor(text=[text], images=imgs, videos=vids,
                           padding=True, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            gen = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen)]
        out = processor.batch_decode(trimmed, skip_special_tokens=True)[0]
        return _json(out)
    except torch.cuda.OutOfMemoryError:
        return None                      # пропускаем эту картинку, идём дальше
    finally:
        del inputs
        torch.cuda.empty_cache()

In [ ]:
# 8) Резюм: подготовка папки чанков и чтение уже готовых id из записанных чанков.
#    Готовые id берём из самих чанк-zip'ов (они пишутся атомарно) — это и есть
#    точка восстановления после обрыва.
import zipfile, glob

CHUNK_DIR = os.path.join(OUT_DIR, "chunks")
os.makedirs(CHUNK_DIR, exist_ok=True)

done = set()
existing_chunks = sorted(glob.glob(os.path.join(CHUNK_DIR, "chunk_*.zip")))
for c in existing_chunks:
    try:
        with zipfile.ZipFile(c) as z:
            for line in z.read("metadata.jsonl").decode("utf-8").splitlines():
                line = line.strip()
                if line:
                    done.add(json.loads(line)["id"])
    except Exception as e:
        print("warn: не смог прочитать", c, "->", e)

chunk_idx = len(existing_chunks) + 1
print("уже готово id:", len(done), "| следующий чанк:", chunk_idx)

In [ ]:
# 9) Основной цикл: OCR -> VLM -> схема. Дедуп/резюм по хэшу картинки,
#    сброс на Drive каждые CHUNK_SIZE, NSFW-фильтр.
import hashlib, io
from PIL import Image
from tqdm.auto import tqdm

seen = set(done)          # готовые + новые в этом прогоне
buffer = []               # накопитель текущего чанка: (id, webp_bytes, rec)
preview_buf = []          # первые несколько за этот прогон — для предпросмотра
stats = {"new": 0, "skip_done": 0, "no_json": 0, "unsafe": 0, "bad_img": 0}

def flush_chunk():
    global chunk_idx, buffer
    if not buffer:
        return
    tmp = os.path.join(CHUNK_DIR, ".tmp_chunk.zip")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        meta = []
        for h, b, rec in buffer:
            z.writestr("images/%s.webp" % h, b)
            meta.append(json.dumps(rec, ensure_ascii=False))
        z.writestr("metadata.jsonl", "\n".join(meta) + "\n")
    os.replace(tmp, os.path.join(CHUNK_DIR, "chunk_%05d.zip" % chunk_idx))  # атомарно
    chunk_idx += 1
    buffer = []

for p in tqdm(paths, desc="annotating"):
    try:
        img = Image.open(p).convert("RGB")
    except Exception:
        stats["bad_img"] += 1
        continue
    buf = io.BytesIO()
    img.save(buf, "WEBP", quality=90)
    b = buf.getvalue()
    h = hashlib.md5(b).hexdigest()
    if h in seen:                      # уже обработан ранее или дубль — пропуск без VLM
        stats["skip_done"] += 1
        continue
    try:
        ocr = " ".join(reader.readtext(p, detail=0)).strip()
    except Exception:
        ocr = ""
    d = annotate(p, ocr)
    if not d:
        stats["no_json"] += 1
        continue
    if DROP_UNSAFE and d.get("safe") is False:
        stats["unsafe"] += 1
        seen.add(h)                    # чтобы не гонять его снова при резюме
        continue
    seen.add(h)
    rec = {"id": h, "source": SOURCE,
           "title": str(d.get("title", "")).strip(),
           "image_desc": str(d.get("image_desc", "")).strip(),
           "meaning": str(d.get("meaning", "")).strip(),
           "caption_text": ocr,
           "src": os.path.relpath(p, INPUT_DIR)}
    buffer.append((h, b, rec))
    if len(preview_buf) < 6:
        preview_buf.append((h, b, rec))
    stats["new"] += 1
    if len(buffer) >= CHUNK_SIZE:
        flush_chunk()

flush_chunk()  # финальный неполный чанк
print("готово. новых:", stats["new"], "|", stats)

In [ ]:
# 10) Слить все чанки в один датасет (когда всё обработано)
OUT_ZIP = os.path.join(OUT_DIR, "annotated_memes.zip")
chunks = sorted(glob.glob(os.path.join(CHUNK_DIR, "chunk_*.zip")))
n_img = 0
seen_names = set()
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zout:
    meta_lines = []
    for c in chunks:
        with zipfile.ZipFile(c) as zin:
            for name in zin.namelist():
                if name.startswith("images/") and name not in seen_names:
                    zout.writestr(name, zin.read(name))
                    seen_names.add(name)
                    n_img += 1
                elif name == "metadata.jsonl":
                    meta_lines.append(zin.read(name).decode("utf-8").strip())
    zout.writestr("metadata.jsonl", "\n".join(l for l in meta_lines if l) + "\n")
print("собрано", n_img, "картинок в", OUT_ZIP)

In [ ]:
# 11) Предпросмотр — сначала прогони на MAX_IMAGES=20 и проверь качество
from PIL import Image
import io as _io
from IPython.display import display

sample = preview_buf
if not sample and chunks:                 # если резюм и в этом прогоне новых нет — берём из последнего чанка
    with zipfile.ZipFile(chunks[-1]) as z:
        metas = [json.loads(l) for l in z.read("metadata.jsonl").decode().splitlines() if l.strip()]
        sample = [(m["id"], z.read("images/%s.webp" % m["id"]), m) for m in metas[:6]]

for h, b, rec in sample:
    print("TITLE:", rec["title"])
    print("OCR  :", rec["caption_text"][:120])
    print("DESC :", rec["image_desc"])
    print("MEAN :", rec["meaning"])
    print("-" * 60)
    display(Image.open(_io.BytesIO(b)).resize((300, 300)))